# [SQL 재현] 2023년 의료기관별 시군구별 진료비 분석

## 단계: 01. 데이터 전처리 — SQL 재현
- 목표: PY_01에서 pandas로 진행한 전처리 과정을 SQL 쿼리로 다시 작성하고, 결과가 PY_01과 같은지 대조한다.
- 환경: Jupyter Notebook + sqlite3 (파이썬에 기본 내장된 파일형 데이터베이스)
- 대조 기준: PY_01_Preprocessing.ipynb 실행 결과

### 1.1 환경 설정 및 데이터 로드
#### 1.1-1 CSV 불러오기 및 컬럼명 정리
- 공공데이터포털 CSV를 pandas로 불러온다 (cp949 인코딩).
- 괄호가 들어간 컬럼명 2개는 SQL에서 쓸 때마다 따옴표가 필요하므로 짧게 바꾼다 (SAS v2와 같은 이름).

In [2]:
import sqlite3
import pandas as pd

df = pd.read_csv(r'C:\data\hira_sigungu_2023.csv', encoding='cp949')
df = df.rename(columns={
    '보험자부담금(선별포함)' : '보험자부담금',
    '요양급여비용총액(선별포함)' : '요양급여비용총액'
})

#### 1.1-2 SQLite 데이터베이스에 테이블 저장
- DB 파일(sql_practice.db)에 연결하고, DataFrame을 `hira` 테이블로 저장한다.
- 출력되는 숫자는 저장된 행 수.

In [3]:
conn = sqlite3.connect(r'C:\data\sql_practice.db')
df.to_sql('hira', conn, if_exists='replace', index=False)

251

### 1.2 데이터 구조 및 타입 파악
#### 1.2-1 상위 3행 조회
- pandas `df.head(3)` 대응: `SELECT *` + `LIMIT 3`

In [4]:
q = """
SELECT *
FROM hira
LIMIT 3
"""
pd.read_sql(q, conn)

,진료년도,시도,시군구,환자수,명세서청구건수,입내원일수,보험자부담금,요양급여비용총액
0,2023,서울,강남구,3182688,18499074,20105055,2302379610700,2962352500080
1,2023,서울,강동구,1060832,10934670,12123283,835458119590,1108786307310
2,2023,서울,강서구,1163258,11007256,11812064,699996239970,946311204440


#### 1.2-2 행 수 및 그룹 수 확인
- pandas `df.shape[0]`, `df['시도'].nunique()` 대응: `COUNT(*)`, `COUNT(DISTINCT 열)`
- SAS 70~76행 PROC SQL을 SQLite로 옮긴 것

In [5]:
q = """
SELECT COUNT(*) AS 행수,
       COUNT(DISTINCT 시도) AS 시도수,
       COUNT(DISTINCT 시군구) AS 시군구수
FROM hira
"""
pd.read_sql(q, conn)

,행수,시도수,시군구수
0,251,17,250


#### 1.2-3 시도별 시군구 수
- pandas `df['시도'].unique()` 확인 + 시도별 개수: `GROUP BY` + `COUNT(*)`

In [7]:
q = """
SELECT 시도,
       COUNT(*) AS 시군구수
FROM hira
GROUP BY 시도
ORDER BY 시군구수 DESC
"""
pd.read_sql(q, conn)

,시도,시군구수
0,경기,42
1,서울,25
2,경북,24
3,전남,22
4,경남,22
5,강원,18
6,충남,16
7,부산,16
8,전북,15
9,충북,14


#### 1.2-4 컬럼 타입 확인
- pandas `df.info()` 대응: `PRAGMA table_info(테이블명)`
- PRAGMA는 SQLite 전용 명령 (MySQL에서는 다른 명령을 씀)

In [8]:
pd.read_sql("PRAGMA table_info(hira)", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,진료년도,INTEGER,0,None,0
1,1,시도,TEXT,0,None,0
2,2,시군구,TEXT,0,None,0
3,3,환자수,INTEGER,0,None,0
4,4,명세서청구건수,INTEGER,0,None,0
5,5,입내원일수,INTEGER,0,None,0
6,6,보험자부담금,INTEGER,0,None,0
7,7,요양급여비용총액,INTEGER,0,None,0
